In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

# PatchQGAN Benchmarking
## (reproduce results from Tsang et al., 2023)
For some pytorch versions, a minor fix is necessary in PQWGAN/train.py:
Replace `b1 = 0` with `b1 = 0.0` in line 38.

Use GitHub repository:

In [3]:
!git clone https://github.com/jasontslxd/PQWGAN.git
!cd PQWGAN && git checkout 245ee2c

Cloning into 'PQWGAN'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 14 (delta 0), reused 0 (delta 0), pack-reused 10 (from 1)
Receiving objects: 100% (14/14), 6.76 KiB | 6.76 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Note: switching to '245ee2c'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 245ee2c Create README.md


In [4]:
# check if already in PQWGAN folder
if os.path.basename(os.getcwd()) != "PQWGAN":
    os.chdir("PQWGAN")

In [5]:
trained_models = {

    'patch_qgan_mnist_0_1_2': {
        'model_dir': '../experiments/patch_qgan_models/mnist/012_28p_11l_25bs',
        'checkpoint': 3100,
        'image_folder': "fake_MNIST_PatchQGAN",
    },
    'patch_qgan_fmnist_0_1': {
        'model_dir': '../experiments/patch_qgan_models/fmnist/01_28p_11l_25bs',
        'checkpoint': 3100,
        'image_folder': "fake_FashionMNIST_PatchQGAN",
    },

}

### Train PatchQGAN models
(if needed)

In [6]:
from train import train as patch_gan_train

In [7]:
patch_qgan_train_runs = {
    # --classes=012 --dataset=mnist --patches=28 --layers=11 --qubits=7 --batch_size=25 --out_folder="out/mnist"
    'patch_qgan_mnist_0_1_2': {
        'classes_str': '012',
        'dataset_str': 'mnist',
        'patches': 28,
        'layers': 11,
        'n_data_qubits': 7,
        'batch_size': 25,
        'out_folder': '../experiments/patch_qgan_models/mnist/',
        'qcritic': False
    },
    # --classes=01 --dataset=fmnist --patches=28 --layers=11 --qubits=7 --batch_size=25 --out_folder="out/fmnist"
    'patch_qgan_fmnist_0_1': {
        'classes_str': '01',
        'dataset_str': 'fmnist',
        'patches': 28,
        'layers': 11,
        'n_data_qubits': 7,
        'batch_size': 25,
        'out_folder': '../experiments/patch_qgan_models/fmnist/',
        'qcritic': False,
    },
}

In [8]:
for run_name, run_kwargs in patch_qgan_train_runs.items():

    checkpoint_wanted = trained_models[run_name]['checkpoint']

    # Check if model already trained
    checkpoint_path = os.path.join("..", "experiments", trained_models[run_name]['model_dir'], f"generator-{checkpoint_wanted}.pt")
    if os.path.exists(checkpoint_path):
        print(f"Model {run_name} already trained, skipping training.")
        continue

    print(f"Starting patch-QGAN training for: {run_name}")
    patch_gan_train(**run_kwargs, checkpoint=0, randn=False, patch_shape=[None, None])  # add default args


Model patch_qgan_mnist_0_1_2 already trained, skipping training.
Model patch_qgan_fmnist_0_1 already trained, skipping training.


### Sampling from trained PatchQGAN models

In [9]:
import numpy as np
import torch
from models.QGCC import PQWGAN_CC
from tqdm import tqdm
from PIL import Image
import os

In [10]:
def patch_qgan_sample(model_dir, checkpoint, n_samples, image_folder,
                      patches, n_data_qubits, layers, patch_shape=[None, None], batch_size=25, image_size=28, channels=1, **kwargs):
    ancillas = 1
    qubits = n_data_qubits + ancillas
    latent_dim = qubits

    device = torch.device("cpu")

    gan = PQWGAN_CC(image_size=image_size, channels=channels, n_generators=patches, n_qubits=qubits,
                    n_ancillas=ancillas, n_layers=layers, patch_shape=patch_shape)
    generator = gan.generator.to(device)
    generator.load_state_dict(torch.load(model_dir + f"/generator-{checkpoint}.pt"))

    total_params = sum(p.numel() for p in generator.parameters() if p.requires_grad)
    print("Total number of parameters in the generator: ", total_params)

    imgs = []
    n_batches = n_samples // batch_size
    # set seed for reproducibility
    torch.manual_seed(42)
    for _ in tqdm(range(n_batches)):
        z = torch.rand(batch_size, latent_dim)  # uniform noise
        with torch.no_grad():
            fake_images = generator(z).cpu().numpy()
        imgs.append(fake_images)

    # Concatenate all batches
    imgs = np.concatenate(imgs, axis=0).squeeze()
    # normalize to [0, 255], assume output is in [-1, 1]
    imgs = (imgs + 1) / 2 * 255

    # invert colors for fashion mnist:
    if 'fmnist' in model_dir.lower():
        imgs = 255 - imgs

    # Save as uncompressed PNGs
    image_root = "../images_FID"
    image_path = os.path.join(image_root, image_folder)
    print(f"Writing images to {image_path}")
    os.makedirs(image_path, exist_ok=True)
    for i, img_array in enumerate(imgs[:n_samples]):
        if channels == 1:
            img = Image.fromarray((img_array).astype(np.uint8), mode='L')  # 'L' for 8-bit grayscale
        else:
            img = Image.fromarray((img_array).astype(np.uint8), mode='RGB')  # 'RGB' for color images

        img.save(os.path.join(image_path, f"img_{i}.png"), compress_level=0)

    return imgs

In [11]:
n_samples = 10_000

for model_name, sample_kwargs in trained_models.items():
    print("Sampling for model in:", model_name)
    sampled_images = patch_qgan_sample(n_samples=n_samples, **sample_kwargs, **patch_qgan_train_runs[model_name])

Sampling for model in: patch_qgan_mnist_0_1_2
Total number of parameters in the generator:  7392


100%|██████████| 400/400 [1:42:12<00:00, 15.33s/it]


Writing images to ../images_FID/fake_MNIST_PatchQGAN
Sampling for model in: patch_qgan_fmnist_0_1
Total number of parameters in the generator:  7392


100%|██████████| 400/400 [1:44:21<00:00, 15.65s/it]


Writing images to ../images_FID/fake_FashionMNIST_PatchQGAN
